In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [ ]:
S3_BUCKET = "dados-teste"
S3_BRONZE = f"s3a://{S3_BUCKET}/bronze"
S3_SILVER = f"s3a://{S3_BUCKET}/silver"
S3_GOLD = f"s3a://{S3_BUCKET}/gold"

SPARK_MASTER = "local[*]"
SPARK_APP_NAME = "medallion-pipeline"

print(f"Bronze:  {S3_BRONZE}")
print(f"Silver:  {S3_SILVER}")
print(f"Gold:    {S3_GOLD}")

Bronze:  s3a://dados-teste/bronze
Silver:  s3a://dados-teste/silver
Gold:    s3a://dados-teste/gold


In [2]:
spark = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .master(SPARK_MASTER) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:4566") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.2,com.amazonaws:aws-java-sdk-bundle:1.12.261") \
    .getOrCreate()

26/08/31 23:30:19 WARN Utils: Your hostname, hugo resolves to a loopback address: 127.0.1.1; using 192.168.0.51 instead (on interface wlp2s0)
26/08/31 23:30:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/hugo/Desktop/Project/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hugo/.ivy2/cache
The jars for the packages stored in: /home/hugo/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6c67443f-3aae-4403-9e37-2dc294b72042;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.261 in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.2/hadoop-aws-3.3.2.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.2!hadoop-aws.jar (198ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.261/aws-java-sdk-bundle-1.12.261.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.261!aws-java-sdk-bundle.jar (7146ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.7.Final

In [5]:
# Lê dados do S3
df_bronze = spark.read.csv(f"{S3_BRONZE}/dados.csv", header=True, inferSchema=True)
df_bronze.show()

+---+---------+-----+-----------+
| id|     nome|valor|  categoria|
+---+---------+-----+-----------+
|  1|Produto A|   50|Eletrônicos|
|  2|Produto B|  150|Eletrônicos|
|  3|Produto C|   75|       Casa|
|  4|Produto D|  200|     Livros|
|  5|Produto E|  120|Eletrônicos|
|  6|Produto F|   30|       Casa|
|  7|Produto G|  250|     Livros|
|  8|Produto H|   95|Eletrônicos|
|  9|Produto I|  180|       Casa|
| 10|Produto J|   45|     Livros|
+---+---------+-----+-----------+



In [8]:
# Filtro (valores > 100)
df_filtrado = df_bronze.filter(col('valor') > 100)

print(f"Registros com valor > 100: {df_filtrado.count()}")
df_filtrado.show()

Registros com valor > 100: 5
+---+---------+-----+-----------+
| id|     nome|valor|  categoria|
+---+---------+-----+-----------+
|  2|Produto B|  150|Eletrônicos|
|  4|Produto D|  200|     Livros|
|  5|Produto E|  120|Eletrônicos|
|  7|Produto G|  250|     Livros|
|  9|Produto I|  180|       Casa|
+---+---------+-----+-----------+



In [10]:
df_filtrado.write.mode("overwrite").csv(f"{S3_SILVER}/df_filtrado", header=True)

26/08/31 23:34:52 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/08/31 23:34:52 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


In [11]:
df_silver = spark.read.csv(f"{S3_SILVER}/df_filtrado", header=True, inferSchema=True)


df_silver.show()

+---+---------+-----+-----------+
| id|     nome|valor|  categoria|
+---+---------+-----+-----------+
|  2|Produto B|  150|Eletrônicos|
|  4|Produto D|  200|     Livros|
|  5|Produto E|  120|Eletrônicos|
|  7|Produto G|  250|     Livros|
|  9|Produto I|  180|       Casa|
+---+---------+-----+-----------+

